# What does a softened teacher teach?

Day 26 · Chapter 15 · CPU worked notebook

Read Chapter15 sections15.5–15.7.


In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


### Prediction

Controlled change: reverse the teacher's favored outcome or initialize the student confidently wrong. Evidence boundary: a fitted three-logit distribution does not establish model compression, reasoning transfer, robustness or faster inference.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
from dongxi_llms.distillation_lab import distillation_loss, distill_distribution
result = distill_distribution(temperature=2.)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
history = result["history"]
axes[0].semilogy([r["update"] for r in history], [r["scaled_loss"] for r in history])
axes[0].set(xlabel="Student SGD update", ylabel="T² forward KL")
for i in range(3):
    axes[1].plot([r["update"] for r in history], [r["student_p"][i] for r in history], label=f"Student outcome {i}")
    axes[1].axhline(history[-1]["teacher_p"][i], linestyle=":", color=f"C{i}")
axes[1].set(xlabel="Student SGD update", ylabel="Probability at inference T1")
axes[1].legend()
plt.show()

### Reference explanation

Reading guide: loss uses training temperature2; the displayed probabilities use inference temperature1. Dashed lines are the fixed teacher distribution.

**Exercise modification reference.** This optional loop changes only the teacher ordering while retaining the original zero student initialization. To isolate the other proposed change instead, restore teacher logits `[2., .5, -1.]` and use student logits `[-5., 0., 5.]`; record the chosen intervention separately.

```python
teacher_variant = torch.tensor([[-1., .5, 2.]], dtype=torch.float64, requires_grad=True)
student_variant = torch.nn.Parameter(torch.zeros_like(teacher_variant))
optimizer = torch.optim.SGD([student_variant], lr=.5)
for _ in range(80):
    optimizer.zero_grad(set_to_none=True)
    loss = distillation_loss(student_variant, teacher_variant, temperature=2., scale=True)
    loss.backward()
    optimizer.step()
assert teacher_variant.grad is None
print(float(loss.detach()), student_variant.detach().softmax(-1).tolist())
```

The shown loop reverses the teacher ordering and retains the zero student initialization. The separately described initialization change is a different intervention. Teacher is detached. Do not claim the original 80-update endpoint for this new variant. This concrete CPU variant has not been executed by the unchanged reference cell; it supplies an exercise reference rather than a new measured result.


Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Compare this fixed preview with your prediction and the current plot.

![Saved reference 1: temperature distillation](../figures/chapter-15/day-26-01_temperature_distillation-01.png)

### Prediction

Exercise: compare analytical gradients at T1,T2,T4, with and without the T² multiplier.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Prediction

Predict: does increasing T without T² scaling preserve the gradient magnitude? Can the teacher receive gradients in standard distillation?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
teacher = torch.tensor([[2., .5, -1.]], dtype=torch.float64, requires_grad=True)
for T in (1., 2., 4.):
    for scale in (False, True):
        student = torch.zeros_like(teacher, requires_grad=True)
        loss = distillation_loss(student, teacher, T, scale=scale)
        loss.backward()
        residual = (student.detach()/T).softmax(-1)-(teacher.detach()/T).softmax(-1)
        analytical = residual*(T if scale else 1/T)
        torch.testing.assert_close(student.grad, analytical)
        print("T,scale,gradient:", T, scale, student.grad.tolist())
assert teacher.grad is None
print("Teacher detached; outcome coordinates must use one aligned vocabulary.")

### Reference explanation

At temperature τ, the unscaled student-logit gradient is `(p_τ − q_τ)/τ`; with τ² scaling it is `τ(p_τ − q_τ)`. The exact equality assertion checks both forms at 1, 2 and 4, and `teacher.grad is None` checks the detach boundary. The source argument `T` denotes temperature.
